In [9]:
# 0. Instalação de Dependências
#%pip install -U langgraph langchain-groq langchain-community duckduckgo-search 
#(Se for rodar pela 1ª vez, descomente a linha de cima)

import os
import getpass
from typing import TypedDict, Annotated, Sequence
import operator

# Bibliotecas do LangChain e LangGraph
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END

print("Bibliotecas importadas com sucesso!")

# 1. Configuração da API do Groq
print("\nPor favor, insira sua API Key do Groq:")
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key: ")

# Inicializando o nosso LLM (usando o Llama 3 de 70B)
llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile")

print("\nGroq configurado e LLM inicializado! Ambiente pronto.")

Bibliotecas importadas com sucesso!

Por favor, insira sua API Key do Groq:

Groq configurado e LLM inicializado! Ambiente pronto.


In [10]:
# 2. Definindo a Memória (Estado) e a Ferramenta Personalizada (@tool)
import requests
from langchain_core.tools import tool

# 1. Definindo o "Estado" (A memória compartilhada do nosso grafo)
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 2. Criando a nossa própria ferramenta
@tool
def buscar_dados_hardware(peca: str) -> str:
    """
    Consulta a API pública da Wikipédia para buscar o resumo e especificações 
    de uma peça de hardware (ex: 'AMD_Ryzen', 'GeForce').
    Sempre use esta ferramenta para pesquisar as peças informadas pelo usuário.
    """
    # Formata o termo para a URL (ex: "Ryzen 7" vira "Ryzen_7")
    termo_formatado = peca.replace(" ", "_")
    url = f"https://pt.wikipedia.org/api/rest_v1/page/summary/{termo_formatado}"
    
    try:
        resposta = requests.get(url)
        if resposta.status_code == 200:
            dados = resposta.json()
            return dados.get("extract", "Especificações detalhadas não encontradas no resumo.")
        else:
            return f"Informações sobre '{peca}' não encontradas na base de dados da Wikipédia."
    except Exception as e:
        return f"Erro ao acessar a API: {str(e)}"

# 3. Substituindo a ferramenta antiga pela nossa nova
tools = [buscar_dados_hardware]

# 4. "Ensinando" o nosso LLM a usar a ferramenta
llm_with_tools = llm.bind_tools(tools)

print("Memória do Grafo criada e Ferramenta Customizada (@tool) inicializada!")
print(f"Ferramentas disponíveis: {[t.name for t in tools]}")

Memória do Grafo criada e Ferramenta Customizada (@tool) inicializada!
Ferramentas disponíveis: ['buscar_dados_hardware']


In [14]:
# 3. Definindo o Comportamento dos Agentes

def criar_no_agente(state: AgentState, system_prompt: str, nome_agente: str):
    messages = list(state["messages"])
    mensagem_sistema = {
        "role": "system",
        "content": system_prompt
    }
    messages.insert(0, mensagem_sistema)
    resposta = llm_with_tools.invoke(messages)
    return {"messages": [resposta]}

# --- PERSONAS (System Prompts) ---

PROMPT_PESQUISADOR = """
Você é o 'Pesquisador de Hardware', um especialista em coletar especificações técnicas.
Sua função é usar a ferramenta 'buscar_dados_hardware' para pesquisar na Wikipédia os componentes informados pelo usuário.
REGRA DE OURO: Pesquise cada peça apenas UMA VEZ. Se a ferramenta retornar que a informação não foi encontrada, NÃO TENTE pesquisar de novo com nomes diferentes. 
Apenas encerre sua participação enviando uma mensagem para o Consultor informando quais peças o usuário deseja e instruindo o Consultor a usar o próprio conhecimento interno para avaliar.
Não dê opiniões técnicas.
"""

PROMPT_CONSULTOR = """
Você é o 'Consultor Sênior de Hardware e Performance'.
Sua função é analisar as especificações técnicas trazidas pelo Pesquisador e avaliar o setup.
Responda diretamente ao usuário com:
1. Resumo técnico das peças (Use seu conhecimento se o Pesquisador não encontrou).
2. Avaliação de compatibilidade.
3. Análise de Gargalo (Bottleneck): A CPU escolhida consegue empurrar a GPU? Ou a GPU é fraca demais para a CPU?
4. Um veredito final claro (Ex: 'Recomendo a compra' ou 'Você terá gargalo').
Seja direto, técnico, mas fácil de entender.
"""

print("Cérebros dos agentes e personas atualizados!")

Cérebros dos agentes e personas atualizados!


In [15]:
# 4. Construindo o Grafo (O Fluxo de Trabalho)
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode

# 1. Funções "Embrulhadas" para cada nó do grafo
def no_pesquisador(state: AgentState):
    return criar_no_agente(state, PROMPT_PESQUISADOR, "Pesquisador")

def no_consultor(state: AgentState):
    return criar_no_agente(state, PROMPT_CONSULTOR, "Consultor")

# 2. Criando a função de "Roteamento" (O guarda de trânsito)
def roteador_pesquisador(state: AgentState):
    """Verifica se o Pesquisador pediu para usar a ferramenta de busca ou se já terminou."""
    ultima_mensagem = state["messages"][-1]
    
    # Se o LLM gerou um pedido de ferramenta, mandamos para o nó da ferramenta
    if getattr(ultima_mensagem, 'tool_calls', None):
        return "ferramentas"
    
    # Se não tem pedido de ferramenta, o trabalho do pesquisador acabou, vai pro consultor
    return "consultor"

# 3. Inicializando o construtor do Grafo com a nossa Memória
workflow = StateGraph(AgentState)

# 4. Adicionando os "Nós" (As estações de trabalho)
workflow.add_node("pesquisador", no_pesquisador)
workflow.add_node("consultor", no_consultor)
workflow.add_node("ferramentas", ToolNode(tools)) # Nó especial que executa a @tool

# 5. Desenhando as setas (Edges) do fluxo
# Começo -> Vai pro Pesquisador
workflow.add_edge(START, "pesquisador")

# O Pesquisador é avaliado pelo Roteador (Ferramenta OU Consultor?)
workflow.add_conditional_edges("pesquisador", roteador_pesquisador)

# Se usou a ferramenta, a resposta da Wikipédia VOLTA pro Pesquisador ler
workflow.add_edge("ferramentas", "pesquisador")

# O Consultor recebe os dados, dá o veredito e encerra (END) o programa
workflow.add_edge("consultor", END)

# 6. Compilando a aplicação final
app = workflow.compile()

print("Grafo desenhado e Aplicação compilada com sucesso!")

Grafo desenhado e Aplicação compilada com sucesso!


In [16]:
# 5. Executando a Aplicação
from langchain_core.messages import HumanMessage

print("--- INICIANDO CONSULTORIA DE HARDWARE ---\n")

# A pergunta do usuário inserida no sistema
pergunta_usuario = "Tenho um PC com um processador Ryzen 7 7800X3D e uma placa de vídeo GTX 960. Quero saltar dessa GTX 960 para uma RX 6600. Dá certo ou vai dar muito gargalo de CPU ou GPU?"

print(f"Usuário: {pergunta_usuario}\n")
print("-" * 50)

# Inicializando o estado da memória com a pergunta do usuário
estado_inicial = {"messages": [HumanMessage(content=pergunta_usuario)]}

# Rodando o grafo passo a passo (streaming) para vermos a colaboração na tela
for output in app.stream(estado_inicial, {"recursion_limit": 10}):
    # O output nos diz qual nó do grafo (agente) acabou de rodar
    for node_name, state_update in output.items():
        print(f"\n[Executando: {node_name.upper()}]")
        
        # Pegando a última mensagem gerada por esse nó
        ultima_mensagem = state_update["messages"][-1]
        
        # Se a mensagem tiver texto, imprimimos para ver o agente "pensando"
        if ultima_mensagem.content:
            print(ultima_mensagem.content)
        
        # Se o agente decidiu acionar a @tool, mostramos o que ele está buscando
        if getattr(ultima_mensagem, 'tool_calls', None):
            chamada = ultima_mensagem.tool_calls[0]
            print(f">> Acionando ferramenta @tool na Wikipédia por: '{chamada['args']['peca']}'")
            
print("\n" + "=" * 50)
print("Consulta de hardware finalizada com sucesso!")

--- INICIANDO CONSULTORIA DE HARDWARE ---

Usuário: Tenho um PC com um processador Ryzen 7 7800X3D e uma placa de vídeo GTX 960. Quero saltar dessa GTX 960 para uma RX 6600. Dá certo ou vai dar muito gargalo de CPU ou GPU?

--------------------------------------------------

[Executando: PESQUISADOR]
>> Acionando ferramenta @tool na Wikipédia por: 'Ryzen 7 7800X3D'

[Executando: FERRAMENTAS]
Informações sobre 'RX 6600' não encontradas na base de dados da Wikipédia.

[Executando: PESQUISADOR]
O usuário deseja saber se o processador Ryzen 7 7800X3D e a placa de vídeo RX 6600 darão gargalo de CPU ou GPU. 
O Consultor deve usar seu conhecimento interno para avaliar a compatibilidade entre esses componentes.

[Executando: CONSULTOR]
 
O Ryzen 7 7800X3D é um processador de alta performance da AMD, com 8 núcleos e 16 threads, e uma frequência turbo de 4,3 GHz. 
A RX 6600 é uma placa de vídeo da AMD, com 8 GB de memória GDDR6 e uma frequência de clock de 2044 MHz. 
A GTX 960 é uma placa de víd